# AI coding Assistant

Help you with writing code comments, unittest, and code fixing.


In [1]:
# packages
import os
import gradio as gr
from enum import Enum
from dotenv import load_dotenv
from openai import OpenAI

In [13]:
# Providers

# Load environment variables
load_dotenv()


class Provider(str, Enum):
    OPENROUTER = "OpenRouter"
    OPENAI = "OpenAI"
    OLLAMA = "Ollama"


clients = {
    Provider.OPENROUTER: OpenAI(
        api_key=os.getenv("OPENROUTER_API_KEY"),
        base_url="https://openrouter.ai/api/v1"
    ),
    Provider.OPENAI: OpenAI(api_key=os.getenv("OPENAI_API_KEY")),
    Provider.OLLAMA: OpenAI(
        api_key="ollama",
        base_url="http://localhost:11434/v1"
    ),
}

models = {
    Provider.OPENROUTER: ["openai/gpt-4o-mini", "mistralai/mistral-7b-instruct"],
    Provider.OPENAI: ["gpt-4o-mini", "gpt-4.1-mini"],
    Provider.OLLAMA: ["llama3", "deepseek-coder"],
}

In [9]:
# Tasks
TASK_OPTIONS = [
    "Add Comments",
    "Explain Code",
    "Write Unit Tests",
    "Fix Code"
]


def build_prompt(tasks, code):
    instructions = []

    if "Add Comments" in tasks:
        instructions.append("Add helpful comments to the code.")
    if "Explain Code" in tasks:
        instructions.append("Explain what the code does.")
    if "Write Unit Tests" in tasks:
        instructions.append("Write comprehensive unit tests.")
    if "Fix Code" in tasks:
        instructions.append("Fix any issues in the code.")

    system_prompt = "You are a senior software engineer."

    user_prompt = f"""
Tasks:
- {"\n- ".join(instructions)}

Code:
{code}
"""

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

In [10]:
# Generation
def run_code_assistant(provider, model, tasks, code):
    if not code.strip():
        return "Please provide code."

    client = clients[Provider(provider)]

    messages = build_prompt(tasks, code)

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0.3,
    )

    return response.choices[0].message.content

In [14]:
# UI
with gr.Blocks(title="AI Coding Assistant") as ui:
    gr.Markdown("# Help you with writing code comments, unittest, and code fixing.")
    gr.Markdown("Perform tasks on your code using different LLM providers.")

    with gr.Row():
        provider = gr.Dropdown(
            choices=[p.value for p in Provider],
            value=Provider.OPENROUTER.value,
            label="Provider"
        )

        model = gr.Dropdown(
            choices=models[Provider.OPENROUTER],
            value=models[Provider.OPENROUTER][0],
            label="Model"
        )

    def update_models(selected_provider):
        return gr.Dropdown(
            choices=models[Provider(selected_provider)],
            value=models[Provider(selected_provider)][0]
        )

    provider.change(update_models, provider, model)

    tasks = gr.CheckboxGroup(
        choices=TASK_OPTIONS,
        value=["Add Comments"],
        label="Tasks"
    )

    code_input = gr.Code(
        label="Code",
        lines=20
    )

    run_button = gr.Button("Run")

    output = gr.Markdown()

    run_button.click(
        run_code_assistant,
        inputs=[provider, model, tasks, code_input],
        outputs=output
    )

ui.launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
